In [2]:
import pandas as pd
from datetime import datetime
import yfinance as yf

roi = pd.read_csv('../data/processed/02_roi_with_macro.csv')

end_date = datetime(2026, 1, 1)
start_date = datetime(2016, 1, 1)
tickers = ["BTC-EUR", "TLT", "EXHA.DE"]

In [3]:
cols_to_drop = [col for col in roi.columns if any(t in col for t in tickers)]
if cols_to_drop:
    roi = roi.drop(columns=cols_to_drop)

data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

prices = data['Close'].copy()

prices_quarterly = prices.resample('QE').last()

prices_quarterly.index = prices_quarterly.index.to_period('Q')
roi['period'] = pd.PeriodIndex(roi['period'], freq='Q')
roi = roi.set_index('period')

print(prices.head())
print(roi.head())

Ticker         BTC-EUR     EXHA.DE        TLT
Date                                         
2016-01-01  400.012909         NaN        NaN
2016-01-02  399.187683         NaN        NaN
2016-01-03  396.360016         NaN        NaN
2016-01-04  400.194977  130.474457  90.984558
2016-01-05  401.898010  130.667267  90.617500
        cpi_index    EUNL.DE        ^GSPC  house_price_index  \
period                                                         
2016Q1  94.000000  35.990002  2059.739990              103.9   
2016Q2  94.966667  37.139999  2098.860107              106.9   
2016Q3  95.466667  38.709999  2168.270020              108.8   
2016Q4  95.400000  42.180000  2238.830078              110.4   
2017Q1  95.500000  44.070000  2362.719971              110.9   

        new_construction_index  existing_property_index  EUNL.DE_norm  \
period                                                                  
2016Q1                   104.2                    103.8    100.000000   
2016Q2    

In [4]:
merged = roi.join(prices_quarterly, how='left')

base_period = pd.Period('2016Q1', freq='Q')

for col in tickers:
    first_val = merged.loc[base_period, col]
    merged[col + '_norm'] = (merged[col] / first_val) * 100

merged = merged.reset_index()
merged['period'] = merged['period'].dt.to_timestamp('Q')

print(merged.head())

      period  cpi_index    EUNL.DE        ^GSPC  house_price_index  \
0 2016-03-31  94.000000  35.990002  2059.739990              103.9   
1 2016-06-30  94.966667  37.139999  2098.860107              106.9   
2 2016-09-30  95.466667  38.709999  2168.270020              108.8   
3 2016-12-31  95.400000  42.180000  2238.830078              110.4   
4 2017-03-31  95.500000  44.070000  2362.719971              110.9   

   new_construction_index  existing_property_index  EUNL.DE_norm  ^GSPC_norm  \
0                   104.2                    103.8    100.000000  100.000000   
1                   106.2                    107.0    103.195326  101.899275   
2                   107.5                    109.0    107.557647  105.269113   
3                   109.5                    110.5    117.199217  108.694791   
4                   109.3                    111.2    122.450674  114.709623   

   house_price_index_norm  new_construction_index_norm  \
0              100.000000               

In [5]:
merged.to_csv('../data/processed/02_roi_with_macro.csv')